# 1.关于流式调用

举例

In [ ]:
from langchain.chat_models import init_chat_model
import os
from dotenv import load_dotenv
from openai import base_url

# 加载配置文件
load_dotenv(override = True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

# 获取大模型
model = init_chat_model(
    model = "deepseek-flash",
    api_key = DEEPSEEK_API_KEY,
    base_url = DEEPSEEK_BASE_URL,
)

In [ ]:
for chunk in model.stream("帮我解释一下什么是Agent?"):
    print(chunk.text, end = "", flush = True)

# 2.批量调用 batch

举例1：一次性接收所有的响应

In [ ]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里?"
]

res = model.batch(messages)

for res in res:
    print(res.content)
    print()

举例2：按照完成的顺序接收响应（谁先简单完成谁先输出）

In [13]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里?"
]

res = model.batch_as_completed(messages)

for res in res:
    print(res)
    print()

(2, AIMessage(content='中国首都是**北京**。', additional_kwargs={'refusal': None, 'reasoning_content': '我们需要回答中文问题。用户问“中国首都在哪里?” 简单。应回答北京。可以简洁。注意中国首都为北京。可以加一句中华人民共和国首都为北京。不需要复杂。 final? We can final.'}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 35, 'total_tokens': 89, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 46, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 35}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '5126b877-455c-40fd-88d0-2884c1001202', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a11c25-bcf7-7b12-8992-6846bd8f4c5b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 35, 'output_tokens': 54, 'total_tokens': 89, 'input_token_details': {'cache_read': 0}, 'out

举例3：性能对比

In [14]:
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪",
]

import time

start = time.time()
res = model.batch(inputs)
batch_time = time.time() - start

print("调用结果是:")
for i, res in enumerate(res):
    print(f"{i + 1}. {res.content}")
print(f"耗时：{batch_time:.2f}秒\n")

调用结果是:
1. Spring has come.
2. Summer is very hot.
3. Autumn leaves falling
4. It snows in winter.
耗时：2.04秒



作为对比，演示循环调用invoke()

In [17]:
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪",
]

import time

start = time.time()
loop_res = []
for inp in inputs:
    res = model.invoke(inp)
    loop_res.append(res)
loop_time = time.time() - start

for i, res in enumerate(loop_res):
    print(f"{i + 1}. {res.content}")

print(f"循环调用耗时:{loop_time:.2f}")

print(f"批量调用节省：{((loop_time - batch_time) / loop_time * 100):.1f}%")

1. Spring has come.
2. Summer is very hot.
3. Autumn leaves fall.
4. It snows in winter.
循环调用耗时:4.16
批量调用节省：51.0%
